# Fine-to-Coarse Sequential Emotion Recognition — Kaggle pipeline

Source comes from the attached private dataset (`f2c-src`). Requires **GPU** + **Internet**
(GoEmotions from the Hugging Face Hub, MELD CSVs from GitHub).

Stages run in order and each writes to `/kaggle/working/{checkpoints,results}`, which Kaggle
keeps as the kernel output. Set `STAGES` below to run a subset.

In [ ]:
import os, shutil, subprocess, sys, time, zipfile, pathlib

WORK = "/kaggle/working"
STAGES = ["stage1", "M1", "M2", "M3", "M4"]   # edit to run a subset
EXTRA = []                                     # e.g. ["--set", "epochs=3"]

INPUT_ROOT = pathlib.Path("/kaggle/input")
NEEDED = {"src", "scripts", "configs"}
ZIPPED = {f"{name}.zip" for name in NEEDED}


def find_source_root(root: pathlib.Path = INPUT_ROOT, max_depth: int = 6) -> pathlib.Path:
    """Locate the attached dataset directory holding src/ scripts/ configs/.

    Kaggle mounts datasets at different depths (historically /kaggle/input/<slug>,
    now /kaggle/input/datasets/<owner>/<slug>/...), so search rather than assume.
    """
    if not root.exists():
        raise FileNotFoundError("/kaggle/input does not exist -- no inputs attached")
    for current, directories, files in os.walk(root):
        depth = len(pathlib.Path(current).relative_to(root).parts)
        if depth > max_depth:
            directories[:] = []
            continue
        names = set(directories) | set(files)
        if NEEDED <= names or ZIPPED <= names:
            return pathlib.Path(current)
    listing = "\n".join(f"  {p}" for p in sorted(root.rglob("*"))[:40]) or "  (empty)"
    raise FileNotFoundError(
        "no attached dataset contains src/ scripts/ configs/. Attach the private dataset "
        f"f2c-src (Add Input > Datasets > Your Datasets) and re-run. Saw:\n{listing}"
    )


SRC = find_source_root()
print("source root:", SRC)

for name in ("src", "scripts", "configs"):
    target = pathlib.Path(WORK, name)
    if target.exists():
        shutil.rmtree(target)
    source_dir = SRC / name
    archive = SRC / f"{name}.zip"
    if source_dir.is_dir():
        shutil.copytree(source_dir, target)
    else:
        with zipfile.ZipFile(archive) as bundle:
            bundle.extractall(WORK if bundle.namelist()[0].startswith(f"{name}/") else target)

os.chdir(WORK)
sys.path.insert(0, f"{WORK}/src")
print("staged:", sorted(p.name for p in pathlib.Path(WORK).iterdir()))

# nvidia-smi is absent from some Kaggle images even when a GPU is attached, so it is
# only a hint here; torch decides in the next cell.
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout.strip() or "nvidia-smi: no output")
except FileNotFoundError:
    print("nvidia-smi not installed in this image")

In [ ]:
!pip install -q "transformers>=4.38" "datasets>=2.16" "scikit-learn>=1.4" "pyyaml>=6.0" 2>&1 | tail -2

ALLOW_CPU = False   # CPU training will not finish inside a 12-hour session

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif ALLOW_CPU:
    print("WARNING: no GPU, running on CPU")
else:
    raise SystemExit(
        "No GPU attached. In the Kaggle notebook editor set Settings > Accelerator to "
        "GPU T4 x2 (or P100) and check your weekly GPU quota, then re-run. "
        "Set ALLOW_CPU = True to proceed anyway."
    )

In [ ]:
def run(cmd):
    """Stream a pipeline step and fail loudly so the kernel stops on error."""
    print("\n>>>", " ".join(cmd), flush=True)
    started = time.time()
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        if "it/s]" in line or "s/it]" in line:      # collapse tqdm spam
            continue
        print(line, end="", flush=True)
    code_ = process.wait()
    print(f"<<< exit={code_} after {(time.time() - started) / 60:.1f} min", flush=True)
    if code_ != 0:
        raise RuntimeError(f"step failed: {' '.join(cmd)}")

## 1. Data

In [ ]:
run([sys.executable, "scripts/prepare_data.py", "--dataset", "all"])

## 2. Stage I — GoEmotions, 27 fine-grained classes

In [ ]:
if "stage1" in STAGES:
    run([sys.executable, "scripts/train_stage1.py", "--config", "configs/stage1.yaml", *EXTRA])

## 3. Stage II — systems M1-M4

In [ ]:
for system in [s for s in STAGES if s.startswith("M")]:
    run([sys.executable, "scripts/train_stage2.py", "--system", system, *EXTRA])

## 4. Tables and error analysis

In [ ]:
run([sys.executable, "scripts/build_results_table.py"])
run([sys.executable, "scripts/error_analysis.py"])

In [ ]:
import json

for path in sorted(pathlib.Path("results").glob("*/metrics.json")):
    payload = json.loads(path.read_text())
    if payload.get("stage") != "II":
        continue
    test = payload["test"]
    print(f"{payload['system']:>4}  acc={test['accuracy']:.4f}  "
          f"macro-F1={test['macro_f1']:.4f}  weighted-F1={test['weighted_f1']:.4f}")

## 5. Trim the output

Kaggle caps kernel output at 20 GB and uploads everything in `/kaggle/working`. Model weights are
large and re-trainable; metrics, predictions and tables are what the report needs.

In [ ]:
KEEP_CHECKPOINTS = False   # True keeps the Stage-I backbone for reuse (~500 MB)

for junk in ("src", "scripts", "configs", "data", "__pycache__"):
    shutil.rmtree(pathlib.Path(WORK, junk), ignore_errors=True)
if not KEEP_CHECKPOINTS:
    shutil.rmtree(pathlib.Path(WORK, "checkpoints"), ignore_errors=True)

for path in sorted(pathlib.Path(WORK).rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:8.2f} MB  {path.relative_to(WORK)}")